Part 1: Distributed Data Processing with Spark

In [48]:
import urllib.request
import os
#pyspark requires older versions of python and java and brew doesn't automatically change the default java version on Mac so the next four cells are just trying to force the correct versions
#and Download the parquet file
%pip install pandas pyarrow

JAVA11 = "/opt/homebrew/opt/openjdk@11/libexec/openjdk.jdk/Contents/Home"

os.environ["JAVA_HOME"] = JAVA11
os.environ["PATH"] = JAVA11 + "/bin:" + os.environ["PATH"]

print("JAVA_HOME:", os.environ["JAVA_HOME"])

def download_taxi_data(url, output_path):
    """Downloads a file if it doesn't already exist and prints the size."""
    if not os.path.exists(output_path):
        print(f"Downloading: {url}...")
        urllib.request.urlretrieve(url, output_path)
        
        size_mb = os.path.getsize(output_path) / 1e6
        print(f"Done! Downloaded: {size_mb:.1f} MB")
    else:
        size_mb = os.path.getsize(output_path) / 1e6
        print(f"File already exists: {output_path} ({size_mb:.1f} MB)")

# --- Configuration ---
DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

# File URLs
files_to_download = {
    "January 2023": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet",
    "February 2023": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet"
}

# --- Execution ---
for month, url in files_to_download.items():
    filename = url.split('/')[-1]
    path = os.path.join(DATA_DIR, filename)
    
    print(f"--- Processing {month} ---")
    download_taxi_data(url, path)




Note: you may need to restart the kernel to use updated packages.
JAVA_HOME: /opt/homebrew/opt/openjdk@11/libexec/openjdk.jdk/Contents/Home
--- Processing January 2023 ---
File already exists: data/yellow_tripdata_2023-01.parquet (47.7 MB)
--- Processing February 2023 ---
File already exists: data/yellow_tripdata_2023-02.parquet (47.7 MB)


In [ ]:
import sys
import pyspark
os.environ["JAVA_HOME"] = "/opt/homebrew/Cellar/openjdk@11/11.0.30/libexec/openjdk.jdk/Contents/Home" 
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)
sys.path.append(os.path.join(os.environ["SPARK_HOME"], "python"))

In [50]:
import subprocess

print("PYTHON:", sys.executable)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("JAVA VERSION:", subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)

PYTHON: /Users/ryanb/3610A3/.venv311/bin/python
JAVA_HOME: /opt/homebrew/Cellar/openjdk@11/11.0.30/libexec/openjdk.jdk/Contents/Home


JAVA VERSION: openjdk version "11.0.30" 2026-01-20
OpenJDK Runtime Environment Homebrew (build 11.0.30+0)
OpenJDK 64-Bit Server VM Homebrew (build 11.0.30+0, mixed mode)



Spark Environment Setup & Data Loading

In [51]:

from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .master('local[*]') \
    .appName('A3') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()


print(f'Spark version: {spark.version}')
print(f'App name: {spark.sparkContext.appName}')
print(f'Master: {spark.sparkContext.master}')
print(f'Default parallelism: {spark.sparkContext.defaultParallelism}')

Spark version: 3.5.1
App name: A3
Master: local[*]
Default parallelism: 11


In [52]:
#Load the NYC DataFrame
df = spark.read.parquet('data/yellow_tripdata_2023-01.parquet')
#Display Schema
df.printSchema()

# Report total row count and Partition count
print(f'Number of rows: {df.count():,}')
print(f'Number of columns: {len(df.columns)}')
print(f'Number of partitions: {df.rdd.getNumPartitions()}')


root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)

Number of rows: 3,066,766
Number of columns: 19
Number of partitions: 11


In [53]:
#Comapring load times
import time
import pandas as pd
# Time Spark read (lazy - just metadata)
start = time.time()
df_spark = spark.read.parquet('data/yellow_tripdata_2023-01.parquet')
spark_read_time = time.time() - start
# Time Spark action (forces full read)
start = time.time()
spark_count = df_spark.count()
spark_action_time = time.time() - start
# Time Pandas read
start = time.time()
df_pandas = pd.read_parquet('data/yellow_tripdata_2023-01.parquet')
pandas_read_time = time.time() - start
print(f'Spark schema read: {spark_read_time:.3f}s (lazy - no data loaded)')
print(f'Spark count action: {spark_action_time:.3f}s ({spark_count:,} rows)')
print(f'Pandas full read: {pandas_read_time:.3f}s ({len(df_pandas):,} rows)')
print(f'Pandas memory usage: {df_pandas.memory_usage(deep=True).sum() / 1e6:.1f}MB')
# Clean up Pandas DataFrame to free memory
del df_pandas

Spark schema read: 0.071s (lazy - no data loaded)
Spark count action: 0.127s (3,066,766 rows)
Pandas full read: 0.479s (3,066,766 rows)
Pandas memory usage: 469.5MB


Pandas functions are not as well paralallized i.e optimized for multicore systems as Spark. This Parallelizaiton also allows the for multi computer processing.

Task 1.2: Data Cleaning & Feature Engineering in Spark

In [ ]:
from pyspark.sql import functions as F

pre_filter_count = df.count()

df_1 = df.dropna(subset=['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'fare_amount', 'trip_distance', 'PULocationID', 'DOLocationID'])
print(f"{df.count() - df_1.count()} Rows Deleted")

df_2 = df_1.filter(
    (F.col('trip_distance') > 0) &
    (F.col('fare_amount') >= 0) & 
    (F.col('total_amount') > 0) &
    (F.col('passenger_count') > 0) &
    (F.col('fare_amount') <= 500) &
    (F.col('tpep_pickup_datetime') < F.col('tpep_dropoff_datetime'))
)
print(f"{df_1.count() - df_2.count()} Rows Deleted")

df_3 = df_2.withColumn(
    'trip_duration_minutes', 
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 60
)

df_4 = df_3.filter(F.col('trip_duration_minutes') > 0)
print(f"{df_3.count() - df_4.count()} Rows Deleted")

df_5 = df_4.withColumn('trip_speed_mph', 
    F.when(F.col('trip_duration_minutes') > 0, 
           F.col('trip_distance') / (F.col('trip_duration_minutes') / 60))
    .otherwise(0)
)

df_6 = df_5.withColumn('pickup_day_of_week', F.dayofweek(F.col('tpep_pickup_datetime')))

df_7 = df_6.withColumn('pickup_hour', F.hour(F.col('tpep_pickup_datetime')))

df_final = df_7.withColumn('tip_percentage',
    F.when(F.col('fare_amount') > 0, (F.col('tip_amount') / F.col('fare_amount')) * 100)
    .otherwise(0)
)

0 Rows Deleted
182618 Rows Deleted
0 Rows Deleted


Task 1.3 Task 1.3: Spark SQL Analytics

In [55]:
#Query 1
df_final.createOrReplaceTempView('taxi_trips')

# 2. Query 1: Top 10 Busiest Hours
query_1 = """
SELECT 
    pickup_hour,
    COUNT(*) AS num_trips,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
FROM taxi_trips
GROUP BY pickup_hour
ORDER BY num_trips DESC
LIMIT 10
"""

q1_results = spark.sql(query_1)
q1_results.show()

# The busiest hours are typically during the evening rush hour. 
# While trip volume is high, the average tip percentage remains relatively consistent across these peak times.

+-----------+---------+--------+------------------+
|pickup_hour|num_trips|avg_fare|avg_tip_percentage|
+-----------+---------+--------+------------------+
|         18|   203600|   16.98|             22.52|
|         17|   197224|   18.62|              21.9|
|         15|   185455|   19.29|             19.35|
|         16|   184665|   19.53|             21.79|
|         19|   182680|   17.65|              22.4|
|         14|   181024|   19.75|             19.27|
|         13|   168766|   18.46|             19.38|
|         12|   160230|   17.75|             19.51|
|         20|   157377|   18.02|              22.9|
|         21|   153379|    18.5|             21.75|
+-----------+---------+--------+------------------+



The busiest hours are typically during the evening rush hour. While trip volume is high, the average tip percentage remains relatively consistent across these peak times.

In [56]:
# Query 2
query_2 = """
SELECT
    pickup_day_of_week,
    ROUND(AVG(trip_speed_mph), 2) AS avg_trip_speed,
    ROUND(AVG(trip_distance), 2) AS avg_trip_distance,
    ROUND(AVG(trip_duration_minutes), 2) AS avg_duration_minutes
FROM taxi_trips
GROUP BY pickup_day_of_week
ORDER BY avg_trip_speed DESC
"""

q2_results = spark.sql(query_2)
q2_results.show()

+------------------+--------------+-----------------+--------------------+
|pickup_day_of_week|avg_trip_speed|avg_trip_distance|avg_duration_minutes|
+------------------+--------------+-----------------+--------------------+
|                 1|         15.37|             3.89|               15.44|
|                 2|         14.51|             3.88|               15.56|
|                 3|         13.79|              3.4|               15.85|
|                 7|         12.81|             3.17|               15.24|
|                 6|         12.41|             3.33|               16.11|
|                 5|         12.38|              3.4|               16.46|
|                 4|         12.13|             3.27|                15.8|
+------------------+--------------+-----------------+--------------------+



Sundays typically show the highest average trip speeds, likely due to reduced traffic congestion. Despite higher speeds, the average trip distance remains comparable to weekdays.

In [57]:
#Query 3
query_3 = """
WITH LocationRevenue AS (
    -- Step 1: Sum revenue per day per location
    SELECT 
        pickup_day_of_week,
        PULocationID,
        SUM(total_amount) AS total_daily_revenue
    FROM taxi_trips
    GROUP BY pickup_day_of_week, PULocationID
),
RankedLocations AS (
    -- Step 2 & 3: Partition by day, order by revenue, and rank
    SELECT 
        pickup_day_of_week,
        PULocationID,
        total_daily_revenue,
        DENSE_RANK() OVER (
            PARTITION BY pickup_day_of_week 
            ORDER BY total_daily_revenue DESC
        ) as revenue_rank
    FROM LocationRevenue
)
-- Step 4: Filter for top 5
SELECT 
    CASE pickup_day_of_week
        WHEN 1 THEN 'Sunday' WHEN 2 THEN 'Monday' WHEN 3 THEN 'Tuesday'
        WHEN 4 THEN 'Wednesday' WHEN 5 THEN 'Thursday' WHEN 6 THEN 'Friday'
        WHEN 7 THEN 'Saturday'
    END AS day_name,
    PULocationID,
    ROUND(total_daily_revenue, 2) AS total_revenue,
    revenue_rank
FROM RankedLocations
WHERE revenue_rank <= 5
ORDER BY pickup_day_of_week, revenue_rank
"""
q3_results = spark.sql(query_3)
q3_results.show(35)


+---------+------------+-------------+------------+
| day_name|PULocationID|total_revenue|revenue_rank|
+---------+------------+-------------+------------+
|   Sunday|         132|   2066472.55|           1|
|   Sunday|         138|    844250.35|           2|
|   Sunday|          79|    393697.61|           3|
|   Sunday|         230|    376818.96|           4|
|   Sunday|         186|     349872.3|           5|
|   Monday|         132|   2140388.19|           1|
|   Monday|         138|    995648.43|           2|
|   Monday|         161|    397160.07|           3|
|   Monday|         186|    368183.27|           4|
|   Monday|         237|    364582.49|           5|
|  Tuesday|         132|   1845968.47|           1|
|  Tuesday|         138|    981013.75|           2|
|  Tuesday|         161|     591064.9|           3|
|  Tuesday|         237|    505243.17|           4|
|  Tuesday|         236|    489424.69|           5|
|Wednesday|         132|   1388934.94|           1|
|Wednesday| 

PickupLocationID 132 seems to be the busiest location by far regardless of day of the week

In [ ]:

query_4 = """
WITH HourlyTrips AS (
    -- Step 1: Get total trips per hour
    SELECT 
        pickup_hour,
        COUNT(*) as trips_in_hour
    FROM taxi_trips
    GROUP BY pickup_hour
),
RunningTotals AS (
    -- Step 2: Calculate cumulative count using a Window Function
    SELECT 
        pickup_hour,
        trips_in_hour,
        SUM(trips_in_hour) OVER (ORDER BY pickup_hour) as cumulative_trips,
        SUM(trips_in_hour) OVER () as total_daily_trips -- Total trips in the whole dataset
    FROM HourlyTrips
)
-- Step 3: Display results and identify the 50% mark
SELECT 
    pickup_hour,
    trips_in_hour,
    cumulative_trips,
    ROUND((cumulative_trips / total_daily_trips) * 100, 2) as cumulative_percentage
FROM RunningTotals
ORDER BY pickup_hour
"""

q4_results = spark.sql(query_4)
q4_results.show(24)



26/04/01 23:05:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/01 23:05:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/01 23:05:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/01 23:05:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/01 23:05:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/01 23:05:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/01 2

+-----------+-------------+----------------+---------------------+
|pickup_hour|trips_in_hour|cumulative_trips|cumulative_percentage|
+-----------+-------------+----------------+---------------------+
|          0|        79565|           79565|                 2.76|
|          1|        55565|          135130|                 4.69|
|          2|        38728|          173858|                 6.03|
|          3|        25053|          198911|                  6.9|
|          4|        15834|          214745|                 7.45|
|          5|        16141|          230886|                 8.01|
|          6|        39940|          270826|                 9.39|
|          7|        79551|          350377|                12.15|
|          8|       107898|          458275|                15.89|
|          9|       122642|          580917|                20.14|
|         10|       135282|          716199|                24.83|
|         11|       145261|          861460|                29

26/04/01 23:05:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/01 23:05:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/01 23:05:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/01 23:05:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Hour 15 is the hour where it the commulative percentage surpass 50%

In [59]:
#Query 5
query_5 = """
SELECT 
    CASE 
        WHEN trip_distance < 2 THEN 'Short (<2 miles)'
        WHEN trip_distance >= 2 AND trip_distance <= 10 THEN 'Medium (2-10 miles)'
        WHEN trip_distance > 10 THEN 'Long (>10 miles)'
    END AS trip_category,
    COUNT(*) as trip_count,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(tip_percentage), 2) AS avg_tip_percentage
FROM taxi_trips
GROUP BY trip_category
ORDER BY avg_distance ASC
"""

q5_results = spark.sql(query_5)
q5_results.show()

+-------------------+----------+--------+------------+------------------+
|      trip_category|trip_count|avg_fare|avg_distance|avg_tip_percentage|
+-------------------+----------+--------+------------+------------------+
|   Short (<2 miles)|   1578479|    9.63|        1.16|             22.97|
|Medium (2-10 miles)|   1069787|   21.55|        3.96|             18.38|
|   Long (>10 miles)|    235882|   64.59|       16.74|             21.88|
+-------------------+----------+--------+------------+------------------+



Most Trips are short
Short Trips have the highest tip as a percentage probably because of lower avg fare cost so cheaper tips make up a bigger percentage
However long trips have a somewhat similar tip percentage making them the by far the highest in terms of tip revenue generation most likely because customers who take these long trips with 
premium rideshare services like Uber tend to skew wealthier less fortunate people would prefer public transit which is relatively robust
Medium has the lowest as a percentage by far proabably due to mix of higher and middle class consumers





Task 1.4: Performance Evaluation

In [ ]:
import time
from pyspark.sql import functions as F


def multi_query(df):
    q1 = df.groupBy('pickup_hour').agg(F.avg('fare_amount')).count()
    q2 = df.groupBy('PULocationID').agg(F.sum('total_amount')).count()
    q3 = df.filter(F.col('trip_distance') > 10).count()
    return q1 + q2 + q3


print("Running Baseline (No Cache)...")
no_cache_times = []
for i in range(3):
    start = time.time()
    multi_query(df_final)
    no_cache_times.append(time.time() - start)

no_cache_time = sum(no_cache_times) / len(no_cache_times)
print(f'Without caching: {no_cache_time:.3f}s (avg of 3 runs)')


df_final.cache() 

print("\nMaterializing Cache...")
start = time.time()
multi_query(df_final)
cache_first_time = time.time() - start
print(f'First run (materializing cache): {cache_first_time:.3f}s')


print("Running with Cache...")
cache_times = []
for i in range(3):
    start = time.time()
    multi_query(df_final)
    cache_times.append(time.time() - start)

cache_avg_time = sum(cache_times) / len(cache_times)
print(f'Cached runs: {cache_avg_time:.3f}s (avg of 3 runs)')
print(f'\nSpeedup from caching: {no_cache_time/cache_avg_time:.2f}x')

Running Baseline (No Cache)...
Without caching: 0.433s (avg of 3 runs)

Materializing Cache...


26/04/01 23:05:17 WARN CacheManager: Asked to cache already cached data.


First run (materializing cache): 0.273s
Running with Cache...
Cached runs: 0.334s (avg of 3 runs)

Speedup from caching: 1.30x


In [ ]:
output_dir = 'data/trips_by_hour'


df_final.select(
    'tpep_pickup_datetime', 'pickup_hour', 'pickup_day_of_week',
    'PULocationID', 'DOLocationID',
    'trip_distance', 'trip_duration_minutes',
    'fare_amount', 'tip_amount', 'total_amount', 'tip_percentage'
).write \
.mode('overwrite') \
.partitionBy('pickup_hour') \
.parquet(output_dir)

print("Verifying directory structure:")
if os.path.exists(output_dir):
    for item in sorted(os.listdir(output_dir)):
        if item.startswith('pickup_hour'):
            path = os.path.join(output_dir, item)
            files = os.listdir(path)
            parquet_files = [f for f in files if f.endswith('.parquet')]
            print(f"  {item}: found {len(parquet_files)} parquet files")
            if '17' in item: 
                print(f"  Example Partition (Hour 17) ready for pruning test.")



start_time = time.time()
df_hour_17 = spark.read.parquet(f"{output_dir}/pickup_hour=17")
count_17 = df_hour_17.count()
print(f"  Rows in Hour 17: {count_17:,}")
print(f"  Read time for single partition: {time.time() - start_time:.4f}s")

Verifying directory structure:
  pickup_hour=0: found 1 parquet files
  pickup_hour=1: found 1 parquet files
  pickup_hour=10: found 1 parquet files
  pickup_hour=11: found 1 parquet files
  pickup_hour=12: found 1 parquet files
  pickup_hour=13: found 1 parquet files
  pickup_hour=14: found 1 parquet files
  pickup_hour=15: found 1 parquet files
  pickup_hour=16: found 1 parquet files
  pickup_hour=17: found 1 parquet files
  Example Partition (Hour 17) ready for pruning test.
  pickup_hour=18: found 1 parquet files
  pickup_hour=19: found 1 parquet files
  pickup_hour=2: found 1 parquet files
  pickup_hour=20: found 1 parquet files
  pickup_hour=21: found 1 parquet files
  pickup_hour=22: found 1 parquet files
  pickup_hour=23: found 1 parquet files
  pickup_hour=3: found 1 parquet files
  pickup_hour=4: found 1 parquet files
  pickup_hour=5: found 1 parquet files
  pickup_hour=6: found 1 parquet files
  pickup_hour=7: found 1 parquet files
  pickup_hour=8: found 1 parquet files
  pi

In [ ]:

df_partitioned = spark.read.parquet(output_dir)
morning_trips = df_partitioned.filter(F.col('pickup_hour') == 17)
morning_trips.explain(mode='formatted')
start = time.time()
morning_trips.count()
partitioned_time = time.time() - start
start = time.time()
df_final.filter(F.col('pickup_hour') == 17).count()
full_scan_time = time.time() - start

=== Execution plan (should show PartitionFilters) ===
== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [11]: [tpep_pickup_datetime#27431, pickup_day_of_week#27432, PULocationID#27433L, DOLocationID#27434L, trip_distance#27435, trip_duration_minutes#27436, fare_amount#27437, tip_amount#27438, total_amount#27439, tip_percentage#27440, pickup_hour#27441]
Batched: true
Location: InMemoryFileIndex [file:/Users/ryanb/3610A3/data/trips_by_hour]
PartitionFilters: [isnotnull(pickup_hour#27441), (pickup_hour#27441 = 17)]
ReadSchema: struct<tpep_pickup_datetime:timestamp_ntz,pickup_day_of_week:int,PULocationID:bigint,DOLocationID:bigint,trip_distance:double,trip_duration_minutes:double,fare_amount:double,tip_amount:double,total_amount:double,tip_percentage:double>

(2) ColumnarToRow [codegen id : 1]
Input [11]: [tpep_pickup_datetime#27431, pickup_day_of_week#27432, PULocationID#27433L, DOLocationID#27434L, trip_distance#27435, trip_duration_minutes#27436, fa

Part 2: RAG Pipeline over Transportation Documents

In [63]:
%pip install pypdf langchain langchain-community chromadb openai \
sentence-transformers matplotlib numpy

Note: you may need to restart the kernel to use updated packages.


Task 2.1: Document Collection & Ingestion

In [ ]:
#Setting up Models and creating docs/
import os
from openai import OpenAI


DOCS_DIR = "docs/"
CHROMA_PATH = "./chroma_db"
LLM_BASE_URL = "https://synapse.sergiomathurin.com/v1"
LLM_MODEL = "llama3.3-70b-instruct"


try:
    with open("API.txt", "r") as file:
        LLM_API_KEY = file.read().strip()
    print("API Key loaded successfully.")
except FileNotFoundError:
    print("ERROR: API.txt not found. Please create it and paste your key inside.")


client = OpenAI(
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY,
)
print(f"LLM Client configured for model: {LLM_MODEL}")

API Key loaded successfully.
LLM Client configured for model: llama3.3-70b-instruct


In [65]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Load PDFs using the global DOCS_DIR variable
loader = PyPDFDirectoryLoader(DOCS_DIR)
raw_documents = loader.load()

total_pages = len(raw_documents)
total_chars = sum(len(doc.page_content) for doc in raw_documents)

print(f"Total PDF pages extracted: {total_pages}")
print(f"Total character count: {total_chars}")

Total PDF pages extracted: 110
Total character count: 238426


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Split text
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = text_splitter.split_documents(raw_documents)

# Generate Embeddings
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Store in ChromaDB using the global CHROMA_PATH variable
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=CHROMA_PATH,
    collection_name="nyc_transport_docs"
)
print(f"Indexed {len(chunks)} chunks in ChromaDB")
print("\n--- Document Descriptions ---")
print("1. Annual_Report_2023.pdf: Gives a breakdown of keyinsights through out the year from data connected by the NYC TLC")
print("2. Mobility-Report-Singlepage-2019: [Insert a brief 1-sentence description of the document]")
print("3. [Insert Filename 3.pdf]: [Insert a brief 1-sentence description of the document]")
print("4. [Insert Filename 4.pdf]: [Insert a brief 1-sentence description of the document]")
print("5. [Insert Filename 5.pdf]: [Insert a brief 1-sentence description of the document]")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7668.78it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indexed 633 chunks in ChromaDB


In [67]:

def format_context(docs):
    """Format retrieved documents into a numbered context string."""
    context_parts = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page", "?")
        context_parts.append(f"[Source {i}: {source}, Page {page}]\n{doc.page_content}")
    return "\n\n---\n\n".join(context_parts)


RAG_PROMPT = """You are a helpful assistant that answers questions based on the provided context. Follow these rules:
1. Only answer based on the provided context.
2. If the context does not contain enough information, say "I cannot answer this based on the provided documents."
3. Cite your sources using [Source N] notation in your answer.
4. Be concise and accurate.

Context:
{context}

Question: {question}

Answer:"""


def ask_and_cite(question, vectorstore, k=3):
    """Retrieve, Augment, Generate, and display detailed citations."""
    print(f"\n{'='*70}")
    print(f"QUESTION: {question}")
    print(f"{'='*70}")
    

    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(question)
    

    context = format_context(docs)
    prompt = RAG_PROMPT.format(context=context, question=question)
    

    response = client.chat.completions.create(
        model=LLM_MODEL, 
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=500,
        temperature=0.2 
    )
    answer = response.choices[0].message.content
    
  
    print(f"\nGENERATED ANSWER:\n{answer}")
    print(f"\n--- RETRIEVED SOURCES & CONTEXT ---")
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "Unknown")
        page = doc.metadata.get("page", "?")
        excerpt = doc.page_content.replace('\n', ' ')[:200]
        print(f"\n[Source {i}]: {source}, Page {page}")
        print(f"Excerpt: {excerpt}...")
        
    return answer, docs

test_questions_task_2_3 = [
    "What are the major trends in NYC traffic speeds according to the mobility report?",
    "How does the Taxi Improvement Fund (TIF) financially support drivers?",
    "What is the current financial health and average price of a taxi medallion?",
    "How many wheelchair accessible vehicles are currently operating?",
    "What are the trends in NYC transit ridership and bus lane expansions?"
]

for q in test_questions_task_2_3:
    ask_and_cite(q, vectorstore=vectorstore, k=3)


QUESTION: What are the major trends in NYC traffic speeds according to the mobility report?

GENERATED ANSWER:
According to the mobility report, the major trend in NYC traffic speeds is a decline in speed in the central business district (CBD) and the Midtown Core, although at a lesser degree than in past years [Source 1, Source 2].

--- RETRIEVED SOURCES & CONTEXT ---

[Source 1]: docs/mobility-report-singlepage-2019.pdf, Page 6
Excerpt: the subway may be nearing capacity. Analyses of data from the screenlines and bridges across New York were added to this  year’s report to provide information about citywide traffic and transportation...

[Source 2]: docs/mobility-report-singlepage-2019.pdf, Page 6
Excerpt: the subway may be nearing capacity. Analyses of data from the screenlines and bridges across New York were added to this  year’s report to provide information about citywide traffic and transportation...

[Source 3]: docs/mobility-report-singlepage-2019.pdf, Page 6
Excerpt: and an

In [68]:
import json

def route_query(query, client):
    system_prompt = """
    You are a query router for a transportation analytics system. 
    Classify the user's query into one of three categories:
    - DATA: Questions about NYC Taxi structured trip data (fares, distances, times, tips).
    - DOCUMENT: Questions about transportation policies, TLC rules, or transit research.
    - HYBRID: Questions requiring both structured data and document policy context.
    
    If the query is ambiguous, default to HYBRID.
    
    Respond ONLY with valid JSON in the following format:
    {"category": "DATA" | "DOCUMENT" | "HYBRID", "reasoning": "Brief explanation of why"}
    """
    
    response = client.chat.completions.create(
        model="llama3.3-70b-instruct", 
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0
    )
    
    raw = response.choices[0].message.content.strip()
    
    # Strip markdown if present (from Lab 5)
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
        
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"category": "HYBRID", "reasoning": "Fallback due to JSON parsing error."}

In [ ]:
def handle_data_query(query, spark, client, error_feedback=""):
    # 1. Generate SQL
    sql_prompt = f"""
    You are a PySpark SQL expert. Write a Spark SQL query to answer the user's question.
    The data is in a temporary view named 'taxi_data'.
    Here is the schema: [INSERT SCHEMA DETAILS HERE]
    
    Respond ONLY with the raw SQL query. Do not include markdown formatting or explanations.
    
    User Question: {query}
    {error_feedback}
    """
    
    sql_response = client.chat.completions.create(
        model="llama3.3-70b-instruct",
        messages=[{"role": "user", "content": sql_prompt}],
        temperature=0
    )
    generated_sql = sql_response.choices[0].message.content.strip()
    
    try:
        results_df = spark.sql(generated_sql)
        raw_results = results_df.limit(10).toPandas().to_dict('records')
        
    except Exception as e:
        if not error_feedback:
            retry_message = f"The previous SQL query failed with this error: {str(e)}. Please rewrite the query to fix it."
            return handle_data_query(query, spark, client, error_feedback=retry_message)
        else:
            return "Failed to execute query after retry."


    synthesis_prompt = f"""
    The user asked: "{query}"
    The database returned the following results: {raw_results}
    Write a natural language response answering the user's question based on these results.
    """
    
    final_response = client.chat.completions.create(
        model="llama3.3-70b-instruct",
        messages=[{"role": "user", "content": synthesis_prompt}],
        temperature=0.2
    )
    
    return final_response.choices[0].message.content, generated_sql, raw_results

In [ ]:
def process_query(user_query, spark, vectorstore, client):
    print(f"--- Processing Query: '{user_query}' ---\n")
    
    routing_decision = route_query(user_query, client) 
    category = routing_decision.get("category", "HYBRID")
    print(f"Classification: {category}")
    print(f"Reasoning: {routing_decision.get('reasoning')}\n")
    

    if category == "DATA":
        final_answer, sql_query, raw_results = handle_data_query(user_query, spark, client)
        print(f"Generated SQL: {sql_query}")
        print(f"Raw Results: {raw_results[:2]}...") 
        print(f"\nFinal Answer: {final_answer}")
        
    elif category == "DOCUMENT":
        final_answer, sources = ask_rag(user_query, vectorstore)
        print(f"Retrieved {len(sources)} chunks.")
        for i, doc in enumerate(sources, 1):
            print(f"Source {i}: {doc.metadata.get('source')} (Page {doc.metadata.get('page')})")
        print(f"\nFinal Answer: {final_answer}")
        
    elif category == "HYBRID":
        final_answer = handle_hybrid_query(user_query, spark, vectorstore, client)
        print(f"\nFinal Answer: {final_answer}")
        
    else:
        print("Unknown category returned by router.")

In [ ]:
def handle_hybrid_query(user_query, spark, vectorstore, client):
    print("Executing HYBRID Pipeline...")
    
    
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    docs = retriever.invoke(user_query)
    doc_context = format_context(docs) 
    print("-> Retrieved document context.")
    
   
    _, sql_query, raw_results = handle_data_query(user_query, spark, client)
    print(f"-> Executed SQL: {sql_query}")
    

    hybrid_prompt = f"""
    You are an expert transportation analyst. Answer the user's question by synthesizing 
    the rules/policies from the Document Context with the empirical numbers from the Data Context.
    
    User Question: {user_query}
    
    Document Context (Policies/Rules):
    {doc_context}
    
    Data Context (Spark SQL Results):
    {raw_results}
    
    Cite your document sources using [Source N] notation and reference the data explicitly.
    """
    
    response = client.chat.completions.create(
        model="llama3.3-70b-instruct",
        messages=[{"role": "user", "content": hybrid_prompt}],
        temperature=0.2
    )
    
    return response.choices[0].message.content